In [ ]:
import csv

# ... [Previous Code] ...

# --- 4. Sorting & Plotting ---
results.sort(key=lambda x: x[0])
samples, jsd_noisy, jsd_unet, jsd_bm4d, jsd_fft, jsd_lr = zip(*results)

# ==========================================
# NEW CODE: Save to CSV
# ==========================================
csv_filename = "denoising_jsd_results.csv"

with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    # Write the header row
    writer.writerow(['Samples', 'Noisy_dmc', 'UNET', 'BM4D', 'FFT', 'Local_Reg'])
    
    # Write the data rows
    for i in range(len(samples)):
        writer.writerow([
            samples[i], 
            jsd_noisy[i], 
            jsd_unet[i], 
            jsd_bm4d[i], 
            jsd_fft[i], 
            jsd_lr[i]
        ])

print(f"Data successfully saved to {csv_filename}!")
# ==========================================

DFT_vs_DMC = D_JS(ref_d, dft_d)
# ... [Rest of Plotting Code] ...

In [4]:
import os
import sys
import glob
import re
import logging
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from bm4d import bm4d

# --- GPU Orchestration ---
# Check for multiple GPUs and map to the second one (index 1) if available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # If 2 or more GPUs exist, use the second one; otherwise, default to the first
        target_gpu = gpus[1] if len(gpus) > 1 else gpus[0]
        tf.config.set_visible_devices(target_gpu, 'GPU')
        print(f"Mapping execution to: {target_gpu}")
    except RuntimeError as e:
        print(f"GPU mapping error: {e}")

# --- Suppress Matplotlib/TF Debug Output ---
logging.getLogger('matplotlib').setLevel(logging.WARNING)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# --- Path Setups ---
qmc_algo_path = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/qmc_algo_tools') 
dev = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/developer_tools') 
sys.path.insert(0, qmc_algo_path) 
sys.path.insert(0, dev) 
from qmc_algo_tools.density_denoise import DensityFourierFilterErrorCeil

# --- Helper Functions ---
def D_JS(p1, p2, tol=1e-16):
    p1 = p1 / np.sum(p1)
    p2 = p2 / np.sum(p2)
    pm = (p1 + p2) / 2
    p1_nonzero = np.abs(p1) > tol
    p2_nonzero = np.abs(p2) > tol
    p1  = np.abs(p1[p1_nonzero])
    pm1 = np.abs(pm[p1_nonzero])
    p2  = np.abs(p2[p2_nonzero])
    pm2 = np.abs(pm[p2_nonzero])
    d = .5 * ( (p1 * np.log(p1 / pm1)).sum() + (p2 * np.log(p2 / pm2)).sum() )
    d /= np.log(2)
    return d

def local_polyfit(y, deg=1, window=3):
    N = len(y)
    half = window // 2
    x = np.arange(N)
    y_out = np.zeros(N)
    for i in range(N):
        start, end = max(0, i - half), min(N, i + half + 1)
        coeffs = np.polyfit(x[start:end], y[start:end], deg)
        y_out[i] = np.polyval(coeffs, x[i])
    return y_out

def transform(density, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return (density - density_ref) / np.sqrt(np.abs(density_ref))
    raise RuntimeError('bad transform type')

def inverse_transform(density_trans, density_ref, transform_type):
    if transform_type == 'residual_noise':
        return density_ref + np.sqrt(np.abs(density_ref)) * density_trans
    raise RuntimeError('bad transform type')

# --- 1. Load VO2 Data ---
dmc_ref_path = '/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/dmc_J2/density_tot_ref_mean.h5'
with h5py.File(dmc_ref_path, 'r') as file:
    ref_d = file['density'][:]

dft_path = '/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5'
with h5py.File(dft_path, 'r') as file:
    dft_d = file['density'][:]

# --- 2. Setup Models ---
unet_model = tf.keras.models.load_model('/pscratch/sd/k/kberard/SCGSR/EDDA/VO2/Density_Models/UNET_3D_Models/residual_denoiser_40M_Blob_NODFT.keras')
data_dir = '/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/dmc_J2/'
files = glob.glob(os.path.join(data_dir, "density_tot_dmc_mix_mean_*.h5"))
dm = DensityFourierFilterErrorCeil(density_ref=dft_d, filter_mode='augment')

results = []
trans_type = 'residual_noise'

# --- 3. Processing Loop ---
print(f"Processing {len(files)} files for VO2...")
for fpath in files:
    match = re.search(r'mean_(\d+)\.h5', os.path.basename(fpath))
    if not match: continue
    sample_count = int(match.group(1))
    
    sigma_psd = 0.000001 * np.sqrt(6881280 / sample_count)
    with h5py.File(fpath, 'r') as f:
        test_d = np.array(f['density'])
    
    # UNET
    input_reshaped = test_d[np.newaxis, ..., np.newaxis]
    dft_reshaped   = dft_d[np.newaxis, ..., np.newaxis]
    input_residual = transform(input_reshaped, dft_reshaped, trans_type)
    pred_residual = unet_model.predict(input_residual, verbose=0)
    denoised_unet = np.maximum(inverse_transform(pred_residual, dft_reshaped, trans_type)[0, ..., 0], 0.0)

    # BM4D
    denoised_bm4d = np.maximum(bm4d(test_d, sigma_psd), 0.0)
    
    # FFT
    denoised_fft = dm.denoise(test_d)

    # Local Regression
    denoised_lr = np.maximum(local_polyfit(test_d.ravel(), deg=2, window=5).reshape(test_d.shape), 0.0)

    results.append({
        "Samples": sample_count,
        "Noisy_dmc": D_JS(test_d, ref_d),
        "UNET": D_JS(denoised_unet, ref_d),
        "BM4D": D_JS(denoised_bm4d, ref_d),
        "FFT": D_JS(denoised_fft, ref_d),
        "Local_Reg": D_JS(denoised_lr, ref_d)
    })

# --- 4. Export & Plotting ---
df = pd.DataFrame(results).sort_values("Samples")
df.to_csv("denoising_performance_results_vo2.csv", index=False)

DFT_vs_DMC = D_JS(ref_d, dft_d)


Mapping execution to: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


I0000 00:00:1778872706.361358 1122891 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38479 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:41:00.0, compute capability: 8.0


Processing 33 files for VO2...


I0000 00:00:1778872708.167699 1131652 service.cc:152] XLA service 0x7f2d5c004040 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778872708.167726 1131652 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2026-05-15 12:18:28.187595: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778872708.377827 1131652 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778872710.584271 1131652 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [5]:
print("here")

here


In [7]:
import os
import matplotlib
matplotlib.use('Agg')  # Prevents backend errors on clusters/SSH
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py

# --- Helper Function for the Baseline ---
def D_JS(p1, p2, tol=1e-16):
    p1 = p1 / np.sum(p1)
    p2 = p2 / np.sum(p2)
    pm = (p1 + p2) / 2
    p1_nonzero = np.abs(p1) > tol
    p2_nonzero = np.abs(p2) > tol
    p1, pm1 = np.abs(p1[p1_nonzero]), np.abs(pm[p1_nonzero])
    p2, pm2 = np.abs(p2[p2_nonzero]), np.abs(pm[p2_nonzero])
    d = .5 * ( (p1 * np.log(p1 / pm1)).sum() + (p2 * np.log(p2 / pm2)).sum() )
    return d / np.log(2)

# --- 1. Load the Data ---
csv_filename = "denoising_performance_results_vo2.csv"
if not os.path.exists(csv_filename):
    print(f"Error: {csv_filename} not found.")
else:
    df = pd.read_csv(csv_filename).sort_values("Samples")

    # --- 2. Recalculate DFT Baseline ---
    # We need the original Ref and DFT h5 files to draw the horizontal line
    dmc_ref_path = '/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/dmc_J2/density_tot_ref_mean.h5'
    dft_path = '/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5'

    with h5py.File(dmc_ref_path, 'r') as f:
        ref_d = f['density'][:]
    with h5py.File(dft_path, 'r') as f:
        dft_d = f['density'][:]

    DFT_vs_DMC = D_JS(ref_d, dft_d)

    # --- 3. Publication Plotting Settings ---
    plt.rcParams.update({
        'font.size': 14, 
        'font.family': 'serif',
        'axes.labelsize': 16,
        'axes.linewidth': 1.5,
        'xtick.major.size': 7,
        'xtick.major.width': 1.5,
        'ytick.major.size': 7,
        'ytick.major.width': 1.5,
        'legend.frameon': True,
        'legend.edgecolor': 'black',
        'legend.fontsize': 12
    })

    fig, ax = plt.subplots(figsize=(10, 8))

    # Plot lines from CSV columns
    ax.plot(df["Samples"], df["Noisy_dmc"], 'r--o', label='Noisy dmc', alpha=0.7, markersize=8)
    ax.plot(df["Samples"], df["UNET"],      'g-s',  label='UNET', linewidth=2.5, markersize=8)
    ax.plot(df["Samples"], df["BM4D"],      'b-^',  label='BM4D', linewidth=2.5, markersize=8)
    ax.plot(df["Samples"], df["FFT"],       'm-D',  label='FFT', linewidth=2.5, markersize=8)
    ax.plot(df["Samples"], df["Local_Reg"], 'c-x',  label='Local Reg', linewidth=2.5, markersize=10)

    # Add the Baseline horizontal line
    ax.axhline(DFT_vs_DMC, color="black", linestyle=":", label="DFT Baseline", linewidth=2)

    # Log Scaling
    ax.set_xscale('log')
    ax.set_yscale('log')

    # Labels and Legibility
    ax.set_xlabel('Number of dmc Samples', fontweight='bold', labelpad=10)
    ax.set_ylabel(r'$D_{JS}$ (Jensen-Shannon Divergence)', fontweight='bold', labelpad=10)
    ax.legend(loc='best')
    ax.grid(False)

    plt.tight_layout()

    # --- 4. Save Outputs ---
    output_base = "vo2_denoising_convergence"
    plt.savefig(f"{output_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_base}.pdf", bbox_inches='tight')

    print(f"Successfully plotted data from {csv_filename}")
    print(f"Files saved: {output_base}.png and {output_base}.pdf")

Successfully plotted data from denoising_performance_results_vo2.csv
Files saved: vo2_denoising_convergence.png and vo2_denoising_convergence.pdf
